# Estimating Token Costs: How Not to Bankrupt Yourself

Before you run an AI model on a big batch of data — 500 customer reviews, a whole spreadsheet, a folder of documents — you should know roughly what it's going to cost. This notebook teaches you to **preview the cost before you spend anything**. There's no free Gemini tier anymore to fall back on if you skip this step, so the habit matters more than it used to.

**Time**: ~15 minutes

**Cost to run this notebook**: effectively $0 — the token-counting call itself has no per-call charge, though it still needs a valid, billed API key to run at all (see Session 1's setup guide).

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../session_1/setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

## Why This Matters

Every API used in this course, Gemini included, now requires a billing method on file before it will issue a key — there's no free, no-card tier to fall back on. Every request costs real money, priced per **token**, not per request. A single test prompt costs a fraction of a cent. A loop that accidentally re-sends a 50-page document on every one of 10,000 iterations can cost real money before you notice, and there's no daily quota wall to stop it early the way a free tier used to.

The habit this notebook builds: **estimate first, run second.**

## What Is a Token?

AI models don't read text as words or letters — they read it as **tokens**, small chunks of text the model was trained to recognize. A token is often a word, but frequently it's a piece of a word.

For example, "marketing" might become two tokens: `market` + `ing`. As a rough rule for English text:

- **1 token ≈ 4 characters**
- **1 token ≈ ¾ of a word**

So a 100-word email is roughly 130–150 tokens. This isn't exact, but it's close enough to sanity-check a cost before you commit to running something.

**Two kinds of tokens, priced differently:**

- **Input tokens** — everything you send: your prompt, any pasted data, instructions.
- **Output tokens** — everything the model generates back. These are usually priced **2–6× higher** than input tokens, because generating text costs more than reading it.

If you're processing many items (rows in a spreadsheet, reviews, emails), your total cost is roughly:

```
number of items × (input tokens per item + output tokens per item) × price per token
```

## Quick Estimate — No API Call Needed

You don't need to call any API to get a ballpark. Run the cell below:

In [ ]:
def estimate_tokens_quick(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English text."""
    return len(text) // 4

sample_review = "Great product, but shipping took way too long. Would still recommend to a friend."
print(f"Estimated tokens: {estimate_tokens_quick(sample_review)}")

Use this the moment you're about to write a loop over a dataset — before you touch the API at all.

## Getting an Exact Count (No Per-Call Charge)

When you want a precise number instead of a rough guess, Gemini offers a **token counting** call that has no per-call charge, because it doesn't generate anything — it still needs a valid, billed `GEMINI_API_KEY` to run, the same one you set up in Colab Secrets during Session 1's setup guide.

Run this cell:

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

review = 'Analyze this customer review for sentiment: "Great product, slow shipping."'

response = client.models.count_tokens(
    model='gemini-2.5-flash-lite',
    contents=review
)
print(f"Exact token count: {response.total_tokens}")

**If you see an error**: the exact method name changes occasionally as the library updates. Search "Gemini API count tokens" for the current syntax. OpenAI and Anthropic have equivalent counting endpoints if you're using a paid key instead.

## Current Pricing (verify before you commit — these change often)

Prices are quoted **per 1 million tokens** and shift over time as providers release new models. Treat the table below as a starting point, not gospel — check the official pricing page before budgeting a large run.

| Provider | Model | Input $/1M tokens | Output $/1M tokens | Notes |
|---|---|---|---|---|
| Google Gemini | Gemini 2.5 Flash-Lite | $0.10 | $0.40 | Cheapest Gemini option |
| Google Gemini | Gemini 3.1 Flash-Lite | $0.25 | $1.50 | Budget-friendly, more capable |
| Google Gemini | Gemini 3.5 Flash | $1.50 | $9.00 | Most capable "Flash" tier |
| OpenAI | GPT-4o mini | $0.15 | $0.60 | Cheapest OpenAI option |
| Anthropic | Claude Haiku 4.5 | $1.00 | $5.00 | Cheapest Claude option |
| Anthropic | Claude Sonnet 5 | $3.00 | $15.00 | Mid-tier, higher quality |

**Official pricing pages** (bookmark these — they're always more current than any table):
- Google Gemini: [ai.google.dev/gemini-api/docs/pricing](https://ai.google.dev/gemini-api/docs/pricing)
- OpenAI: [openai.com/api/pricing](https://openai.com/api/pricing/)
- Anthropic: [claude.com/pricing](https://claude.com/pricing)

⚠️ **Billing required, no free quota.** Every provider above requires a billing method on file before issuing a key, and there's no daily allowance to absorb a mistake — every request is billed from the first one. Rate limits still apply per minute and per day regardless of billing, so check your current limits on each provider's dashboard before a large run, and don't assume a fixed number from any guide (including this one) is still accurate.

## Build Your Own Cost Calculator

Before running anything on a paid tier, estimate the total. Run the cell below, then try changing the numbers to match your own use case:

In [ ]:
def estimate_cost(
    num_items: int,
    avg_input_chars: int,
    avg_output_tokens: int,
    price_per_1m_input: float,
    price_per_1m_output: float,
) -> float:
    """Estimate total USD cost for a batch job, before running it."""
    input_tokens_per_item = avg_input_chars / 4  # rough char-to-token estimate
    total_input_tokens = num_items * input_tokens_per_item
    total_output_tokens = num_items * avg_output_tokens

    input_cost = (total_input_tokens / 1_000_000) * price_per_1m_input
    output_cost = (total_output_tokens / 1_000_000) * price_per_1m_output
    return input_cost + output_cost

cost = estimate_cost(
    num_items=500,
    avg_input_chars=250,       # ~a 50-word review plus your prompt instructions
    avg_output_tokens=40,      # a short sentiment + confidence + key phrases response
    price_per_1m_input=0.15,   # GPT-4o mini
    price_per_1m_output=0.60,
)
print(f"Estimated cost: ${cost:.4f}")

## Worked Example: 500 Customer Reviews

Say you want to run sentiment analysis on 500 customer reviews, each about 50 words (~250 characters), with a short structured response (~40 tokens) per review. Run the cell below to compare models:

In [ ]:
pricing = {
    "Gemini 2.5 Flash-Lite": (0.10, 0.40),
    "GPT-4o mini":           (0.15, 0.60),
    "Claude Haiku 4.5":      (1.00, 5.00),
    "Claude Sonnet 5":       (3.00, 15.00),
}

print(f"{'Model':<24}{'Estimated cost for 500 reviews'}")
print("-" * 55)
for model_name, (price_in, price_out) in pricing.items():
    total = estimate_cost(
        num_items=500,
        avg_input_chars=250,
        avg_output_tokens=40,
        price_per_1m_input=price_in,
        price_per_1m_output=price_out,
    )
    print(f"{model_name:<24}${total:.4f}")

Even the "expensive" option here costs a few cents for 500 reviews. The real risk isn't a single well-planned batch — it's an **unplanned loop**: retrying on every error without a cap, accidentally re-processing the same data twice, or pasting an entire multi-page document into every single request in a loop instead of once.

Estimate the total *before* you press run, especially when the item count is large or your prompts include a lot of pasted context.

## 5 Rules to Avoid Surprise Bills

1. **Estimate before you loop.** Multiply item count × tokens per item × price, *before* writing the `for` loop that calls the API.
2. **Test on the cheapest model first.** Prototype your prompt on the cheapest available model (Flash-Lite, GPT-4o mini, Haiku) — none of them are free anymore, but all of them cost little enough to iterate on freely. Only upgrade to a pricier model if quality genuinely requires it.
3. **Cap your output length.** Most APIs accept a `max_tokens` (or similar) parameter. Set a sane ceiling so a model that starts rambling can't run up an unexpectedly large output bill.
4. **Don't re-send large context repeatedly.** If you're referencing the same big document across many calls, summarize it once instead of pasting the whole thing into every request.
5. **Set a spending limit in your provider dashboard.** OpenAI, Anthropic, and Google Cloud all let you set a hard monthly spending cap or usage alert. Turn one on for every key you use — it's the safety net for when your estimate is wrong.

## Where to Watch Your Real Spending

- **Google Gemini**: [aistudio.google.com/apikey](https://aistudio.google.com/apikey) — shows your key usage and quota
- **OpenAI**: [platform.openai.com/usage](https://platform.openai.com/usage) — usage dashboard and spending limits
- **Anthropic**: [console.anthropic.com](https://console.anthropic.com) — Console → Usage and Cost

Check these *before* a large run, not just after.

## ✅ You're Done

You now know how to:
- Estimate token counts without calling an API
- Get an exact token count, at no per-call charge, when you need precision
- Calculate the projected cost of a batch job before running it
- Set guardrails (cheap models first, output caps, spending limits) so a mistake can't turn into a surprise bill

### Quick Reference

Keep this cell handy — copy it into any notebook where you're about to process a batch of data:

In [ ]:
# Rough token estimate — no API call needed
def estimate_tokens_quick(text: str) -> int:
    return len(text) // 4

# Cost estimate for a batch job — run this before your real loop
def estimate_cost(num_items, avg_input_chars, avg_output_tokens,
                   price_per_1m_input, price_per_1m_output):
    total_input_tokens = num_items * (avg_input_chars / 4)
    total_output_tokens = num_items * avg_output_tokens
    input_cost = (total_input_tokens / 1_000_000) * price_per_1m_input
    output_cost = (total_output_tokens / 1_000_000) * price_per_1m_output
    return input_cost + output_cost

**The habit that matters most**: before running anything against a real dataset, ask "how many items, how many tokens each, at what price?" — and do that multiplication before you do the loop.

**Questions?** Post in the Circle community.